In [1]:
import sympy as sp
import numpy as np
import math
import collections
import tqdm
import itertools
import random

import utils

In [59]:
def gen_v0(bound):
    rv = collections.defaultdict(set)
    for m, n, p, q in tqdm.tqdm(
        itertools.product(
            range(1, bound + 1),
            range(1, bound + 1),
            range(1, bound + 1),
            range(1, bound + 1),
        ),
        total=(bound) ** 4,
    ):
        # if n == 0 or m == 0 or p == 0 or q == 0:
        #     continue
        if n >= m or p >= q or p >= m:
            continue
        a, b, c, d = m * p - n * q, m * q + n * p, m * p + n * q, m * q - n * p
        if a > b or c > d:
            continue
        P = (m * m + n * n) * (p * p + q * q)
        if P % 2 != 0:
            continue
        P = P // 2
        Q = ((m * m - n * n) * (p * p - q * q)) ** 2 + 16 * (m * n * p * q) ** 2
        assert Q % 16 == 0

        Q = Q // 16
        rv[(P, Q)].add(tuple(abs(x) for x in (a, b, c, d)))
    return rv


l = gen_v0(100)
l

100%|██████████| 100000000/100000000 [00:27<00:00, 3657051.12it/s]


defaultdict(set,
            {(25, 72): {(1, 7, 5, 5)},
             (65, 424): {(3, 11, 7, 9)},
             (125, 1492): {(5, 15, 9, 13)},
             (205, 3924): {(7, 19, 11, 17)},
             (305, 8584): {(9, 23, 13, 21)},
             (425, 16552): {(11, 27, 15, 25)},
             (565, 29124): {(13, 31, 17, 29)},
             (725, 47812): {(15, 35, 19, 33)},
             (905, 74344): {(17, 39, 21, 37)},
             (1105, 110664): {(19, 43, 23, 41)},
             (1325, 158932): {(21, 47, 25, 45)},
             (1565, 221524): {(23, 51, 27, 49)},
             (1825, 301032): {(25, 55, 29, 53)},
             (2105, 400264): {(27, 59, 31, 57)},
             (2405, 522244): {(29, 63, 33, 61)},
             (2725, 670212): {(31, 67, 35, 65)},
             (3065, 847624): {(33, 71, 37, 69)},
             (3425, 1058152): {(35, 75, 39, 73)},
             (3805, 1305684): {(37, 79, 41, 77)},
             (4205, 1594324): {(39, 83, 43, 81)},
             (4625, 1928392): {(41, 87,

In [60]:
[x for x in l.values() if len(x) > 1]

[{(11, 367, 131, 343), (77, 359, 101, 353)},
 {(212, 474, 356, 378), (252, 454, 282, 436)},
 {(22, 734, 262, 686), (154, 718, 202, 706)},
 {(424, 948, 712, 756), (504, 908, 564, 872)},
 {(33, 1101, 393, 1029), (231, 1077, 303, 1059)},
 {(636, 1422, 1068, 1134), (756, 1362, 846, 1308)},
 {(44, 1468, 524, 1372), (308, 1436, 404, 1412)},
 {(55, 1835, 655, 1715), (385, 1795, 505, 1765)},
 {(848, 1896, 1424, 1512), (1008, 1816, 1128, 1744)},
 {(66, 2202, 786, 2058), (462, 2154, 606, 2118)},
 {(1060, 2370, 1780, 1890), (1260, 2270, 1410, 2180)},
 {(77, 2569, 917, 2401), (539, 2513, 707, 2471)},
 {(1272, 2844, 2136, 2268), (1512, 2724, 1692, 2616)},
 {(88, 2936, 1048, 2744), (616, 2872, 808, 2824)},
 {(1484, 3318, 2492, 2646), (1764, 3178, 1974, 3052)},
 {(99, 3303, 1179, 3087), (693, 3231, 909, 3177)},
 {(110, 3670, 1310, 3430), (770, 3590, 1010, 3530)},
 {(1696, 3792, 2848, 3024), (2016, 3632, 2256, 3488)},
 {(121, 4037, 1441, 3773), (847, 3949, 1111, 3883)},
 {(1908, 4266, 3204, 3402), (22

In [2]:
m, n, p, q, s, t = sp.var("m, n, p, q, s, t")
i = sp.I
s1, s2, s3 = sp.var("s1, s2, s3")
k1 = m + i * s1 * n
k2 = p + i * s2 * q
k3 = s + i * s3 * t
K = k1 * k2 * k3
K = K.as_poly(i)
Ki, Kr = K.all_coeffs()

conj_perm = list(itertools.product([1], [1, -1], [1, -1]))
full_exprs = [
    (
        Ki.subs(list(zip([s1, s2, s3], conj_perm[i]))),
        Kr.subs(list(zip([s1, s2, s3], conj_perm[i]))),
    )
    for i in range(len(conj_perm))
]
# print(full_exprs)


def validate(pairs):
    p1, p2, p3, p4 = pairs
    P = (m**2 + n**2) * (p**2 + q**2) * (s**2 + t**2)
    assert (P - p1[0] ** 2 - p1[1] ** 2).simplify() == 0
    assert (P - p2[0] ** 2 - p2[1] ** 2).simplify() == 0
    assert (P - p3[0] ** 2 - p3[1] ** 2).simplify() == 0
    assert (P - p4[0] ** 2 - p4[1] ** 2).simplify() == 0


validate(full_exprs)
p1, p2, p3, p4 = full_exprs
P = (m**2 + n**2) * (p**2 + q**2) * (s**2 + t**2)
Q12 = (p1[0] ** 2 - p1[1] ** 2) ** 2 + (p2[0] ** 2 - p2[1] ** 2) ** 2
Q13 = (p1[0] ** 2 - p1[1] ** 2) ** 2 + (p3[0] ** 2 - p3[1] ** 2) ** 2
Q14 = (p1[0] ** 2 - p1[1] ** 2) ** 2 + (p4[0] ** 2 - p4[1] ** 2) ** 2
Q23 = (p2[0] ** 2 - p2[1] ** 2) ** 2 + (p3[0] ** 2 - p3[1] ** 2) ** 2
Q24 = (p2[0] ** 2 - p2[1] ** 2) ** 2 + (p4[0] ** 2 - p4[1] ** 2) ** 2
Q34 = (p3[0] ** 2 - p3[1] ** 2) ** 2 + (p4[0] ** 2 - p4[1] ** 2) ** 2

D1 = Q12 - Q34
D2 = Q13 - Q24
D3 = Q14 - Q23
D1.expand().simplify(), D2.expand().simplify(), D3.expand().simplify()

(32*m*n*p*q*(-m**2*p**2*s**4 + 6*m**2*p**2*s**2*t**2 - m**2*p**2*t**4 + m**2*q**2*s**4 - 6*m**2*q**2*s**2*t**2 + m**2*q**2*t**4 + n**2*p**2*s**4 - 6*n**2*p**2*s**2*t**2 + n**2*p**2*t**4 - n**2*q**2*s**4 + 6*n**2*q**2*s**2*t**2 - n**2*q**2*t**4),
 32*m*n*s*t*(-m**2*p**4*s**2 + m**2*p**4*t**2 + 6*m**2*p**2*q**2*s**2 - 6*m**2*p**2*q**2*t**2 - m**2*q**4*s**2 + m**2*q**4*t**2 + n**2*p**4*s**2 - n**2*p**4*t**2 - 6*n**2*p**2*q**2*s**2 + 6*n**2*p**2*q**2*t**2 + n**2*q**4*s**2 - n**2*q**4*t**2),
 32*p*q*s*t*(-m**4*p**2*s**2 + m**4*p**2*t**2 + m**4*q**2*s**2 - m**4*q**2*t**2 + 6*m**2*n**2*p**2*s**2 - 6*m**2*n**2*p**2*t**2 - 6*m**2*n**2*q**2*s**2 + 6*m**2*n**2*q**2*t**2 - n**4*p**2*s**2 + n**4*p**2*t**2 + n**4*q**2*s**2 - n**4*q**2*t**2))

In [ ]:
D1.expand().simplify()

32*m*n*p*q*(-m**2*p**2*s**4 + 6*m**2*p**2*s**2*t**2 - m**2*p**2*t**4 + m**2*q**2*s**4 - 6*m**2*q**2*s**2*t**2 + m**2*q**2*t**4 + n**2*p**2*s**4 - 6*n**2*p**2*s**2*t**2 + n**2*p**2*t**4 - n**2*q**2*s**4 + 6*n**2*q**2*s**2*t**2 - n**2*q**2*t**4)

In [63]:
D2.expand().simplify()

32*m*n*s*t*(-m**2*p**4*s**2 + m**2*p**4*t**2 + 6*m**2*p**2*q**2*s**2 - 6*m**2*p**2*q**2*t**2 - m**2*q**4*s**2 + m**2*q**4*t**2 + n**2*p**4*s**2 - n**2*p**4*t**2 - 6*n**2*p**2*q**2*s**2 + 6*n**2*p**2*q**2*t**2 + n**2*q**4*s**2 - n**2*q**4*t**2)

In [64]:
D3.expand().simplify()

32*p*q*s*t*(-m**4*p**2*s**2 + m**4*p**2*t**2 + m**4*q**2*s**2 - m**4*q**2*t**2 + 6*m**2*n**2*p**2*s**2 - 6*m**2*n**2*p**2*t**2 - 6*m**2*n**2*q**2*s**2 + 6*m**2*n**2*q**2*t**2 - n**4*p**2*s**2 + n**4*p**2*t**2 + n**4*q**2*s**2 - n**4*q**2*t**2)

In [65]:
sp.groebner([P, D1], order="grevlex")[0].simplify()

n**5*p*q*(p**4*s**6 - 5*p**4*s**4*t**2 - 5*p**4*s**2*t**4 + p**4*t**6 - q**4*s**6 + 5*q**4*s**4*t**2 + 5*q**4*s**2*t**4 - q**4*t**6)

In [66]:
sp.groebner([P, D1], order="grevlex")[1].simplify()

m*n**3*p*q*(p**4*s**6 - 5*p**4*s**4*t**2 - 5*p**4*s**2*t**4 + p**4*t**6 - q**4*s**6 + 5*q**4*s**4*t**2 + 5*q**4*s**2*t**4 - q**4*t**6)

In [67]:
sp.groebner([P, D1], order="grevlex")[2].simplify()

m*n*p*q*(4*m**2*p**4*t**4 - 4*m**2*q**4*t**4 - n**2*p**4*s**4 + 6*n**2*p**4*s**2*t**2 + 3*n**2*p**4*t**4 + n**2*q**4*s**4 - 6*n**2*q**4*s**2*t**2 - 3*n**2*q**4*t**4)

In [68]:
sp.groebner([P, D1], order="grevlex")[3].simplify()

m*n*q*(4*m**2*p**4*t**4 - 4*m**2*p**2*q**2*t**4 + m**2*q**4*s**4 - 6*m**2*q**4*s**2*t**2 - 7*m**2*q**4*t**4 - n**2*p**4*s**4 + 6*n**2*p**4*s**2*t**2 + 3*n**2*p**4*t**4 + n**2*p**2*q**2*s**4 - 6*n**2*p**2*q**2*s**2*t**2 - 3*n**2*p**2*q**2*t**4 + n**2*q**4*s**4 - 6*n**2*q**4*s**2*t**2 - 7*n**2*q**4*t**4)

In [69]:
sp.groebner([P, D1], order="grevlex")[4].simplify()

m*n*p*q*(-4*m**2*p**2*t**4 + m**2*q**2*s**4 - 6*m**2*q**2*s**2*t**2 - 3*m**2*q**2*t**4 + n**2*p**2*s**4 - 6*n**2*p**2*s**2*t**2 - 3*n**2*p**2*t**4 - 4*n**2*q**2*t**4)

In [70]:
sp.groebner([P, D1], order="grevlex")[5].simplify()

m**2*p**2*s**2 + m**2*p**2*t**2 + m**2*q**2*s**2 + m**2*q**2*t**2 + n**2*p**2*s**2 + n**2*p**2*t**2 + n**2*q**2*s**2 + n**2*q**2*t**2

In [ ]:
sp.groebner([D1, P], order="grevlex")[5].simplify()

m**4*p**2 + m**4*q**2 - 2*m**3*n*p**2 - 2*m**3*n*q**2 - 2*m**3*p**2*t - 2*m**3*q**2*t + 2*m**2*n**2*p**2 + 2*m**2*n**2*q**2 + 2*m**2*n*p**2*t + 2*m**2*n*q**2*t + 2*m**2*p**2*t**2 + 2*m**2*q**2*t**2 - 2*m*n**3*p**2 - 2*m*n**3*q**2 - 2*m*n**2*p**2*t - 2*m*n**2*q**2*t + n**4*p**2 + n**4*q**2 + 2*n**3*p**2*t + 2*n**3*q**2*t + 2*n**2*p**2*t**2 + 2*n**2*q**2*t**2

In [47]:
x = D1.simplify()
D1_c = x.as_terms()[-1][-1]
D1_c

list(sp.groebner([D1_c, P]))
# D1_d.subs({s: s+t, m:m+n, p:p+q}).simplify()
# D1_d

[-4*m**2*p**2*t**4 + m**2*q**2*s**4 - 6*m**2*q**2*s**2*t**2 - 3*m**2*q**2*t**4 + n**2*p**2*s**4 - 6*n**2*p**2*s**2*t**2 - 3*n**2*p**2*t**4 - 4*n**2*q**2*t**4,
 m**2*p**2*s**2 + m**2*p**2*t**2 + m**2*q**2*s**2 + m**2*q**2*t**2 + n**2*p**2*s**2 + n**2*p**2*t**2 + n**2*q**2*s**2 + n**2*q**2*t**2,
 4*m**4*p**2*t**4 - m**4*q**2*s**4 + 6*m**4*q**2*s**2*t**2 + 3*m**4*q**2*t**4 - 4*n**4*p**2*t**4 + n**4*q**2*s**4 - 6*n**4*q**2*s**2*t**2 - 3*n**4*q**2*t**4,
 m**4*q**2*s**6 - 5*m**4*q**2*s**4*t**2 - 5*m**4*q**2*s**2*t**4 + m**4*q**2*t**6 - n**4*q**2*s**6 + 5*n**4*q**2*s**4*t**2 + 5*n**4*q**2*s**2*t**4 - n**4*q**2*t**6]

In [48]:
D1_c

-m**2*p**2*s**4 + 6*m**2*p**2*s**2*t**2 - m**2*p**2*t**4 + m**2*q**2*s**4 - 6*m**2*q**2*s**2*t**2 + m**2*q**2*t**4 + n**2*p**2*s**4 - 6*n**2*p**2*s**2*t**2 + n**2*p**2*t**4 - n**2*q**2*s**4 + 6*n**2*q**2*s**2*t**2 - n**2*q**2*t**4

In [65]:
# (m**2 * p**2 )*(-s**4 -t**4) + (m**2 *q **2) *(s**4 + t**4) + (n**2 * p **2)


(s**4 + t**4) * (
    -(m**2) * p**2 + m**2 * q**2 + n**2 * p**2 - n**2 * q**2
) + (6 * s**2 * t**2) * (
    m**2 * p**2 - m**2 * q**2 - n**2 * p**2 + n**2 * q**2
)
k = (-(m**2) * p**2 + m**2 * q**2 + n**2 * p**2 - n**2 * q**2) * (
    s**4 + t**4 - 6 * s**2 * t**2
)
k = (m**2 - n**2) * (q**2 - p**2) * (s**4 + t**4 - 6 * s**2 * t**2)

k

(m**2 - n**2)*(-p**2 + q**2)*(s**4 - 6*s**2*t**2 + t**4)

In [64]:
(k - D1_c).simplify()
# k.expand()

0

In [67]:
D2.simplify()

32*m*n*s*t*(-m**2*p**4*s**2 + m**2*p**4*t**2 + 6*m**2*p**2*q**2*s**2 - 6*m**2*p**2*q**2*t**2 - m**2*q**4*s**2 + m**2*q**4*t**2 + n**2*p**4*s**2 - n**2*p**4*t**2 - 6*n**2*p**2*q**2*s**2 + 6*n**2*p**2*q**2*t**2 + n**2*q**4*s**2 - n**2*q**4*t**2)

In [70]:
k = (m**2 - n**2) * (t**2 - s**2) * (p**4 + q**4 - 6 * p**2 * q**2)
k.expand()

-m**2*p**4*s**2 + m**2*p**4*t**2 + 6*m**2*p**2*q**2*s**2 - 6*m**2*p**2*q**2*t**2 - m**2*q**4*s**2 + m**2*q**4*t**2 + n**2*p**4*s**2 - n**2*p**4*t**2 - 6*n**2*p**2*q**2*s**2 + 6*n**2*p**2*q**2*t**2 + n**2*q**4*s**2 - n**2*q**4*t**2

In [71]:
D3.simplify()

32*p*q*s*t*(-m**4*p**2*s**2 + m**4*p**2*t**2 + m**4*q**2*s**2 - m**4*q**2*t**2 + 6*m**2*n**2*p**2*s**2 - 6*m**2*n**2*p**2*t**2 - 6*m**2*n**2*q**2*s**2 + 6*m**2*n**2*q**2*t**2 - n**4*p**2*s**2 + n**4*p**2*t**2 + n**4*q**2*s**2 - n**4*q**2*t**2)

In [72]:
k = (p**2 - q**2) * (t**2 - s**2) * (m**4 + n**4 - 6 * m**2 * n**2)
k.expand()

-m**4*p**2*s**2 + m**4*p**2*t**2 + m**4*q**2*s**2 - m**4*q**2*t**2 + 6*m**2*n**2*p**2*s**2 - 6*m**2*n**2*p**2*t**2 - 6*m**2*n**2*q**2*s**2 + 6*m**2*n**2*q**2*t**2 - n**4*p**2*s**2 + n**4*p**2*t**2 + n**4*q**2*s**2 - n**4*q**2*t**2

In [71]:
def numerically_validate(pairs_evaluated):
    p1 = pairs_evaluated[0]
    P_m_2 = p1[0] ** 2 + p1[1] ** 2
    # print(f"{P_m_2=}")
    for pair in pairs_evaluated:
        assert P_m_2 == pair[0] ** 2 + pair[1] ** 2


def subs_mnpqst_to_pairs(pairs, mnpqst):
    return [
        tuple(entry.subs([*zip((m, n, p, q, s, t), mnpqst)]) for entry in pair)
        for pair in pairs
    ]


def randomize_numerically_validate(pairs, n_tests=100, bound=1000):
    for _ in tqdm.tqdm(range(n_tests)):
        mnpqst = [random.randint(-bound, bound) for _ in range(6)]
        pairs_evaluated = subs_mnpqst_to_pairs(pairs, mnpqst)
        numerically_validate(pairs_evaluated)


random.seed(0x321)

numerically_validate(subs_mnpqst_to_pairs(full_exprs, (1, 2, 3, 4, 5, 6)))
randomize_numerically_validate(full_exprs)

100%|██████████| 100/100 [00:00<00:00, 149.00it/s]


In [72]:
def randomize_find_quadruplet(n_tests=100, bound=1000):
    sols = collections.defaultdict(list)
    for _ in tqdm.tqdm(range(n_tests)):
        mnpqst = [random.randint(-bound, bound) for _ in range(6)]
        D1_eval = D1.subs([*zip((m, n, p, q, s, t), mnpqst)])
        if D1_eval == 0:
            print(f"D1: {D1_eval} {mnpqst}")
            sols["D1"].append(mnpqst)
        D2_eval = D2.subs([*zip((m, n, p, q, s, t), mnpqst)])
        if D2_eval == 0:
            print(f"D2: {D2_eval} {mnpqst}")
            sols["D2"].append(mnpqst)
        D3_eval = D3.subs([*zip((m, n, p, q, s, t), mnpqst)])
        if D3_eval == 0:
            print(f"D3: {D3_eval} {mnpqst}")
            sols["D3"].append(mnpqst)
    return sols


randomize_find_quadruplet(n_tests=1000)

  0%|          | 0/1000 [00:00<?, ?it/s]

  2%|▏         | 22/1000 [00:01<01:20, 12.19it/s]

D2: 0 [178, 316, -91, 139, 0, -439]
D3: 0 [178, 316, -91, 139, 0, -439]
D1: 0 [0, -830, -729, -661, -850, 909]
D2: 0 [0, -830, -729, -661, -850, 909]


 15%|█▌        | 154/1000 [00:12<01:06, 12.81it/s]

D1: 0 [257, 775, 0, 654, 916, 345]
D3: 0 [257, 775, 0, 654, 916, 345]


...

D1: 0 [307, -307, -837, -486, 52, -352]
D2: 0 [307, -307, -837, -486, 52, -352]


 94%|█████████▍| 944/1000 [01:14<00:04, 13.46it/s]

D1: 0 [-45, -45, -426, -130, 330, -834]
D2: 0 [-45, -45, -426, -130, 330, -834]


100%|██████████| 1000/1000 [01:19<00:00, 12.66it/s]


defaultdict(list,
            {'D2': [[178, 316, -91, 139, 0, -439],
              [0, -830, -729, -661, -850, 909],
              [809, -176, 775, -639, -219, 219],
              [-797, -797, -58, -951, 777, -576],
              [157, -157, -137, -677, -686, 955],
              [-195, -293, 821, 617, -668, 668],
              [-472, 762, -457, -929, -484, 484],
              [307, -307, -837, -486, 52, -352],
              [-45, -45, -426, -130, 330, -834]],
             'D3': [[178, 316, -91, 139, 0, -439],
              [257, 775, 0, 654, 916, 345],
              [809, -176, 775, -639, -219, 219],
              [-195, -293, 821, 617, -668, 668],
              [-472, 762, -457, -929, -484, 484]],
             'D1': [[0, -830, -729, -661, -850, 909],
              [257, 775, 0, 654, 916, 345],
              [-797, -797, -58, -951, 777, -576],
              [157, -157, -137, -677, -686, 955],
              [307, -307, -837, -486, 52, -352],
              [-45, -45, -426, -130, 330, -83

In [174]:
D1

(-(m*p*s - m*q*t - n*p*t - n*q*s)**2 + (m*p*t + m*q*s + n*p*s - n*q*t)**2)**2 - (-(m*p*s - m*q*t + n*p*t + n*q*s)**2 + (-m*p*t - m*q*s + n*p*s - n*q*t)**2)**2 - (-(m*p*s + m*q*t - n*p*t + n*q*s)**2 + (m*p*t - m*q*s + n*p*s + n*q*t)**2)**2 + (-(m*p*s + m*q*t + n*p*t - n*q*s)**2 + (-m*p*t + m*q*s + n*p*s + n*q*t)**2)**2

In [ ]:
# [130, -839, 134, -134, 738, -565]
# [910, 558, -138, 138, -435, 428]
# [-242, -802, 0, -502, -525, 73]
# [0, -830, -729, -661, -850, 909]
# [257, 775, 0, 654, 916, 345]
pairs_evaluated = subs_mnpqst_to_pairs(full_exprs, [130, -839, 134, -134, 738, -565])
numerically_validate(pairs_evaluated)
pairs_evaluated

[(-42147958, -143477418),
 (-149504738, 3248562),
 (-143477418, 42147958),
 (3248562, 149504738)]

# Investigating n = 3

In [9]:
N = 500

hits = collections.defaultdict(list)

for i in tqdm.tqdm(range(1, N)):
    for j in range(1, i):
        hits[i**2 + j**2].append((i, j))

collections.Counter([len(v) for v in hits.values()])

100%|██████████| 499/499 [00:00<00:00, 984.82it/s] 


Counter({1: 46701,
         2: 23879,
         4: 3994,
         3: 2879,
         6: 549,
         8: 129,
         5: 127,
         9: 11,
         7: 9,
         12: 3,
         10: 2})

In [10]:
sols = [
    v for k, v in hits.items() if len(v) == 2 and sum([x**2 for x in v[0]]) % 2 == 0
]


ab, cd = sols[0]
a, b = ab
c, d = cd
print(a, b, c, d)


calc_P = lambda a, b, c, d: (a**2 + b**2) // 2
calc_Q = lambda a, b, c, d: (a**4 + b**4 - 2 * (c**2) * (d**2)) // 4
calc_R = lambda a, b, c, d: ((c**2) * (d**2) - (a**2) * (b**2)) // 2


def verify_quadruplet(a, b, c, d):

    P = calc_P(a, b, c, d)
    Q = calc_Q(a, b, c, d)
    R = calc_R(a, b, c, d)

    print(calc_P(a, b, c, d))
    print(calc_Q(a, b, c, d))
    print(calc_R(a, b, c, d))

    _, is_gem = verify_skew_normal_squareish([P, Q, R**2])
    assert is_gem


verify_quadruplet(a, b, c, d)

for sol in tqdm.tqdm(sols):
    ab, cd = sol
    a, b = ab
    c, d = cd
    verify_quadruplet(a, b, c, d)

9 7 11 3
65
1696
-1440
1696 [1440]
65 [16, 56]
0 [7, 9, 3, 11]
iroots=[-7, 7, -9, 9, -3, 3, -11, 11] 
is_gem=True


  6%|▌         | 678/11884 [00:00<00:01, 6776.97it/s]

65
1696
-1440
1696 [1440]
65 [16, 56]
0 [7, 9, 3, 11]
iroots=[-7, 7, -9, 9, -3, 3, -11, 11] 
is_gem=True
85
4176
-2880
4176 [2880]
85 [36, 84]
0 [7, 11, 1, 13]
iroots=[-7, 7, -11, 11, -1, 1, -13, 13] 
is_gem=True
125
5968
-4032
5968 [4032]
125 [44, 100]
0 [9, 13, 5, 15]
iroots=[-9, 9, -13, 13, -5, 5, -15, 15] 
is_gem=True
145
10656
-10080
10656 [10080]
145 [24, 144]
0 [11, 13, 1, 17]
iroots=[-11, 11, -13, 13, -1, 1, -17, 17] 
is_gem=True
130
10116
-5760
10116 [5760]
130 [66, 126]
0 [8, 14, 2, 16]
iroots=[-8, 8, -14, 14, -2, 2, -16, 16] 
is_gem=True
170
12196
-11520
12196 [11520]
170 [26, 154]
0 [12, 14, 4, 18]
iroots=[-12, 12, -14, 14, -4, 4, -18, 18] 
is_gem=True
185
20896
-10080
20896 [10080]
185 [104, 176]
0 [9, 17, 3, 19]
iroots=[-9, 9, -17, 17, -3, 3, -19, 19] 
is_gem=True
205
15696
-8640
15696 [8640]
205 [84, 156]
0 [11, 17, 7, 19]
iroots=[-11, 11, -17, 17, -7, 7, -19, 19] 
is_gem=True
260
27136
-23040
27136 [23040]
260 [64, 224]
0 [14, 18, 6, 22]
iroots=[-14, 14, -18, 18, -6, 6,

 11%|█▏        | 1356/11884 [00:00<00:01, 6370.80it/s]


116558404 [60278400]
15602 [7502, 13298]
0 [90, 152, 48, 170]
iroots=[-90, 90, -152, 152, -48, 48, -170, 170] 
is_gem=True
15970
148911876
-98017920
148911876 [98017920]
15970 [7134, 15714]
0 [94, 152, 16, 178]
iroots=[-94, 94, -152, 152, -16, 16, -178, 178] 
is_gem=True
16160
96879616
-48660480
96879616 [48660480]
16160 [6944, 12064]
0 [96, 152, 64, 168]
iroots=[-96, 96, -152, 152, -64, 64, -168, 168] 
is_gem=True
16960
161611776
-123863040
161611776 [123863040]
16960 [6144, 16896]
0 [104, 152, 8, 184]
iroots=[-104, 104, -152, 152, -8, 8, -184, 184] 
is_gem=True
17384
90779200
-58060800
90779200 [58060800]
17384 [5720, 12200]
0 [108, 152, 72, 172]
iroots=[-108, 108, -152, 152, -72, 72, -172, 172] 
is_gem=True
17602
47120004
-16848000
47120004 [16848000]
17602 [5502, 7998]
0 [110, 152, 98, 160]
iroots=[-110, 110, -152, 152, -98, 98, -160, 160] 
is_gem=True
18280
132941376
-109670400
132941376 [109670400]
18280 [4824, 15576]
0 [116, 152, 52, 184]
iroots=[-116, 116, -152, 152, -52, 52, 

...

[88, 388, 68, 392]
iroots=[-88, 88, -388, 388, -68, 68, -392, 392] 
is_gem=True
84520
4570030656
-210862080
4570030656 [210862080]
84520 [66024, 69144]
0 [136, 388, 124, 392]
iroots=[-136, 136, -388, 388, -124, 124, -392, 392] 
is_gem=True
85072
5727965184
-1441382400
5727965184 [1441382400]
85072 [65472, 84672]
0 [140, 388, 20, 412]
iroots=[-140, 140, -388, 388, -20, 20, -412, 412] 
is_gem=True
86224
5556326400
-1419264000
5556326400 [1419264000]
86224 [64320, 83520]
0 [148, 388, 52, 412]
iroots=[-148, 148, -388, 388, -52, 52, -412, 412] 
is_gem=True
87440
4862058496
-879943680
4862058496 [879943680]
87440 [63104, 75776]
0 [156, 388, 108, 404]
iroots=[-156, 156, -388, 388, -108, 108, -404, 404] 
is_gem=True
88072
5286533184
-1383782400
5286533184 [1383782400]
88072 [62472, 81672]
0 [160, 388, 80, 412]
iroots=[-160, 160, -388, 388, -80, 80, -412, 412] 
is_gem=True
90760
5173379136
-1599252480
5173379136 [1599252480]
90760 [59784, 82296]
0 [176, 388, 92, 416]
iroots=[-176, 176, -388, 38

 86%|████████▌ | 10191/11884 [00:01<00:00, 10076.55it/s]

[8040, 76440]
0 [392, 412, 292, 488]
iroots=[-392, 392, -412, 412, -292, 292, -488, 488] 
is_gem=True
88097
6939575584
-137944800
6939575584 [137944800]
88097 [82472, 84128]
0 [75, 413, 63, 415]
iroots=[-75, 75, -413, 413, -63, 63, -415, 415] 
is_gem=True
92129
6843536800
-690703200
6843536800 [690703200]
92129 [78440, 86800]
0 [117, 413, 73, 423]
iroots=[-117, 117, -413, 413, -73, 73, -423, 423] 
is_gem=True
94129
6811156800
-968083200
6811156800 [968083200]
94129 [76440, 88200]
0 [133, 413, 77, 427]
iroots=[-133, 133, -413, 413, -77, 77, -427, 427] 
is_gem=True
94669
7188181200
-1427371200
7188181200 [1427371200]
94669 [75900, 92820]
0 [137, 413, 43, 433]
iroots=[-137, 137, -413, 413, -43, 43, -433, 433] 
is_gem=True
95509
7373307600
-1739304000
7373307600 [1739304000]
95509 [75060, 95460]
0 [143, 413, 7, 437]
iroots=[-143, 143, -413, 413, -7, 7, -437, 437] 
is_gem=True
95797
7324280784
-1733428800
7324280784 [1733428800]
95797 [74772, 95172]
0 [145, 413, 25, 437]
iroots=[-145, 145, 

100%|██████████| 11884/11884 [00:01<00:00, 9354.38it/s] 

177817
1852020864
-1443657600
1852020864 [1443657600]
177817 [20208, 57408]
0 [397, 445, 347, 485]
iroots=[-397, 397, -445, 445, -347, 347, -485, 485] 
is_gem=True
101258
9893513764
-356428800
9893513764 [356428800]
101258 [97658, 101242]
0 [60, 446, 4, 450]
iroots=[-60, 60, -446, 446, -4, 4, -450, 450] 
is_gem=True
103508
9815544064
-712857600
9815544064 [712857600]
103508 [95408, 102608]
0 [90, 446, 30, 454]
iroots=[-90, 90, -446, 446, -30, 30, -454, 454] 
is_gem=True
104066
9892749700
-896227200
9892749700 [896227200]
104066 [94850, 103870]
0 [96, 446, 14, 456]
iroots=[-96, 96, -446, 446, -14, 14, -456, 456] 
is_gem=True
105508
9423512064
-698457600
9423512064 [698457600]
105508 [93408, 100608]
0 [110, 446, 70, 454]
iroots=[-110, 110, -446, 446, -70, 70, -454, 454] 
is_gem=True
106658
8848613764
-337075200
8848613764 [337075200]
106658 [92258, 95842]
0 [120, 446, 104, 450]
iroots=[-120, 120, -446, 446, -104, 104, -450, 450] 
is_gem=True
109826
9530474500
-1593446400
9530474500 [1593

# N = 4

In [11]:
x, P, Q, R, S = sp.var("x, P, Q, R, S")
a, b, c, d, e, f, g, h = sp.var("a,b,c,d,e,f,g,h")
p = (((x**2 - P) ** 2 - Q) ** 2 - R) ** 2 - S**2
p_0 = ((((x**2 - P) ** 2 - Q) ** 2 - R) - S).as_poly(x)
p_1 = ((((x**2 - P) ** 2 - Q) ** 2 - R) + S).as_poly(x)

R_ = sp.prod([x**2 - v**2 for v in [a, b, c, d, e, f, g, h]])
R_0 = sp.prod(
    [
        x**2 - v**2
        for v in [
            a,
            b,
            c,
            d,
        ]
    ]
).as_poly(x)
R_1 = sp.prod([x**2 - v**2 for v in [e, f, g, h]]).as_poly(x)

In [12]:
p_0

Poly(x**8 - 4*P*x**6 + (6*P**2 - 2*Q)*x**4 + (-4*P**3 + 4*P*Q)*x**2 + P**4 - 2*P**2*Q + Q**2 - R - S, x, domain='ZZ[P,Q,R,S]')

In [13]:
R_0

Poly(x**8 + (-a**2 - b**2 - c**2 - d**2)*x**6 + (a**2*b**2 + a**2*c**2 + a**2*d**2 + b**2*c**2 + b**2*d**2 + c**2*d**2)*x**4 + (-a**2*b**2*c**2 - a**2*b**2*d**2 - a**2*c**2*d**2 - b**2*c**2*d**2)*x**2 + a**2*b**2*c**2*d**2, x, domain='ZZ[a,b,c,d]')

In [14]:
p = ((x**2 - P) ** 2 - Q) ** 2 - R**2


eqs = []

x0, x1 = sp.var("x0, x1")

eqs.append(sp.Eq(x0**2, Q + R))
eqs.append(sp.Eq(x1**2, Q - R))

x00, x01, x10, x11 = sp.var("x00, x01, x10, x11")


eqs.append(sp.Eq(x00**2, P + x0))
eqs.append(sp.Eq(x01**2, P - x0))
eqs.append(sp.Eq(x10**2, P + x1))
eqs.append(sp.Eq(x11**2, P - x1))

In [ ]:
x0**2 + x1**2 == 2 * Q
x0**2 - x1**2 == 2 * R


x00**2 + x01**2 == 2 * P
x10**2 + x11**2 == 2 * P
x00**2 - x01**2 == 2 * x0
x10**2 - x11**2 == 2 * x1


(x00**2 - x01**2) ** 2 + (x10**2 - x11**2) ** 2 == (2 * x0) ** 2 + (
    2 * x1
) ** 2 == 8 * Q
(x00**2 - x01**2) ** 2 + (x00**2 + x01**2) ** 2 + (x10**2 - x11**2) ** 2 + (
    x10**2 + x11**2
) ** 2 == 8 * Q + 2 * (2 * P) ** 2


K0 = x00**4 + x01**4 + x10**4 + x11**4
K0 == 4 * Q + 4 * P**2
K1 = x00**2 * x01**2 + x10**2 * x11**2
K1 == 2 * P**2 - 2 * Q


y0 = x00 * x01
y1 = x10 * x11
K1 == y0**2 + y1**2
K1 == (x00**2) * (2 * P - x00**2) + (x10**2) * (2 * P - x10**2)
K1 == -((x00**2 - P) ** 2) + P**2 - (x10**2 - P) ** 2 + P**2
2 * Q == 2 * P**2 - K1 == (x00**2 - P) ** 2 + (x10**2 - P) ** 2
x0 == x00**2 - P
x1 == x10**2 - P
K1_ = sp.var("K1_")

In [ ]:
sp.groebner(
    [x00**2 + x01**2 - (x10**2 + x11**2), K1_ - (x00**2 * x01**2 + x10**2 * x11**2)]
)

GroebnerBasis([x00**2 + x01**2 - x10**2 - x11**2, K1_ + x01**4 - x01**2*x10**2 - x01**2*x11**2 - x10**2*x11**2], x00, x01, x10, x11, K1_, domain='ZZ', order='lex')

In [ ]:
sp.groebner(
    [
        x00**2 + x01**2 - 2 * P,
        x10**2 + x11**2 - 2 * P,
        K1_ - (x00**2 * x01**2 + x10**2 * x11**2),
    ]
)

GroebnerBasis([-2*P + x00**2 + x01**2, K1_ - 2*P*x01**2 - 2*P*x11**2 + x01**4 + x11**4, -2*P + x10**2 + x11**2], x00, x01, x10, x11, K1_, P, domain='ZZ', order='lex')

In [ ]:
sp.groebner(
    [
        x00**2 + x01**2 - 2 * P,
        x10**2 + x11**2 - 2 * P,
        (x00**2 - x01**2) ** 2 + (x10**2 - x11**2) ** 2 - 8 * Q,
    ]
)

GroebnerBasis([-2*P + x00**2 + x01**2, 2*P**2 - 2*P*x01**2 - 2*P*x11**2 - 2*Q + x01**4 + x11**4, -2*P + x10**2 + x11**2], x00, x01, x10, x11, P, Q, domain='ZZ', order='lex')

In [ ]:
sp.groebner(
    [
        a**2 + b**2 - 2 * P,
        c**2 + d**2 - 2 * P,
        e**2 + f**2 - 2 * P,
        g**2 + h**2 - 2 * P,
        (a**2 - b**2) ** 2 + (c**2 - d**2) ** 2 - 8 * Q,
        (e**2 - f**2) ** 2 + (g**2 - h**2) ** 2 - 8 * Q,
    ],
    order="grevlex",
)

GroebnerBasis([b**4 + d**4 - 2*P*b**2 - 2*P*d**2 + 2*P**2 - 2*Q, f**4 + h**4 - 2*P*f**2 - 2*P*h**2 + 2*P**2 - 2*Q, a**2 + b**2 - 2*P, c**2 + d**2 - 2*P, e**2 + f**2 - 2*P, g**2 + h**2 - 2*P], a, b, c, d, e, f, g, h, P, Q, domain='ZZ', order='grevlex')

In [30]:
sp.groebner(
    [
        a**2 + b**2 - (g**2 + h**2),
        c**2 + d**2 - (g**2 + h**2),
        e**2 + f**2 - (g**2 + h**2),
        (a**2 - b**2) ** 2
        + (c**2 - d**2) ** 2
        - ((e**2 - f**2) ** 2 + (g**2 - h**2) ** 2),
    ],
    order="grevlex",
)

GroebnerBasis([b**4 + d**4 - f**4 - b**2*g**2 - d**2*g**2 + f**2*g**2 - b**2*h**2 - d**2*h**2 + f**2*h**2 + g**2*h**2, a**2 + b**2 - g**2 - h**2, c**2 + d**2 - g**2 - h**2, e**2 + f**2 - g**2 - h**2], a, b, c, d, e, f, g, h, domain='ZZ', order='grevlex')

In [26]:
sp.groebner(
    [
        a**2 + b**2 - 2 * P,
        c**2 + d**2 - 2 * P,
        e**2 + f**2 - 2 * P,
        g**2 + h**2 - 2 * P,
        (a**2 - b**2) ** 2
        + (c**2 - d**2) ** 2
        - ((e**2 - f**2) ** 2 + (g**2 - h**2) ** 2),
    ]
)

GroebnerBasis([-2*P + a**2 + b**2, -2*P*b**2 - 2*P*d**2 + 2*P*f**2 + 2*P*h**2 + b**4 + d**4 - f**4 - h**4, -2*P + c**2 + d**2, -2*P + e**2 + f**2, -2*P + g**2 + h**2], a, b, c, d, e, f, g, h, P, domain='ZZ', order='lex')

## A dumb but might work approach

In [12]:
hits_10k = hits

In [42]:
N = 5000

hits = collections.defaultdict(list)

for i in tqdm.tqdm(range(1, N)):
    for j in range(1, min(i, int(math.sqrt(N**2 - i**2)))):
        P = i**2 + j**2
        if P % 2 == 0:
            hits[P // 2].append((i, j))

print(collections.Counter([len(v) for v in hits.values()]))


print(f"{len(sols)=}")

# calc_Q_tag = lambda a, b, c, d: ((a**2) * (b**2) + (c**2) * (d**2))


def handle_pairs(pairs):
    pairs_of_pairs = list(itertools.combinations(pairs, 2))
    valid_pair_of_pairs_of_pairs = [
        (p1, p2)
        for p1, p2 in itertools.combinations(pairs_of_pairs, 2)
        if set(p1).isdisjoint(p2)  # ensures pairs are disjoint (no shared elements)
    ]
    for abcdefgh in valid_pair_of_pairs_of_pairs:
        abcd, efgh = abcdefgh
        ab, cd = abcd
        ef, gh = efgh
        a, b = ab
        c, d = cd
        e, f = ef
        g, h = gh
        if calc_Q(a, b, c, d) == calc_Q(e, f, g, h):
            print(
                len(pairs),
                calc_P(a, b, c, d),
                sp.factorint(calc_P(a, b, c, d)),
                calc_Q(a, b, c, d),
                abcdefgh,
            )


def handle_pairs_2(pairs):
    P = (pairs[0][0] ** 2 + pairs[0][1] ** 2) // 2
    norms = [(pair[0] ** 2) * (pair[1] ** 2) for pair in pairs]
    q_tags = collections.defaultdict(list)
    for i, norm_i in enumerate(norms):
        for j, norm_j in enumerate(norms[:i]):
            q_tags[norm_i + norm_j].append((i, j))
    for k, v in q_tags.items():

        if len(v) > 1:
            Q = (2 * P**2 - k) // 2
            E = 8 * P**2 - 2 * k
            print(
                f"{P=}, {sp.factorint(P)}, {Q=}, {sp.factorint(Q)}, {sp.factorint(E)} {len(pairs)=}",
                [(pairs[idxs[0]], pairs[idxs[1]]) for idxs in v],
            )
            if len(v) > 2:
                print("SPECIAL")
            else:
                if not set(v[0]).isdisjoint(set(v[1])):
                    print("NOT DISJOINT")


for P, pairs in tqdm.tqdm(hits.items()):
    handle_pairs_2(pairs)

100%|██████████| 4999/4999 [00:06<00:00, 805.82it/s] 


Counter({1: 1054843, 2: 968483, 4: 279263, 3: 80023, 6: 40492, 8: 27494, 12: 3655, 5: 3115, 9: 1323, 10: 893, 16: 534, 7: 225, 18: 75, 15: 32, 13: 15, 20: 15, 14: 11, 24: 11})
len(sols)=11884


  3%|▎         | 71334/2460502 [00:00<00:03, 713316.87it/s]

P=67405, {5: 1, 13: 1, 17: 1, 61: 1}, Q=3525798096, {2: 4, 3: 2, 17: 1, 1049: 1, 1373: 1}, {2: 2, 17: 1, 474660713: 1} len(pairs)=8 [((359, 77), (353, 101)), ((367, 11), (343, 131))]
P=134810, {2: 1, 5: 1, 13: 1, 17: 1, 61: 1}, Q=4070543716, {2: 2, 17: 1, 313: 1, 191249: 1}, {2: 5, 11: 1, 17: 1, 14869171: 1} len(pairs)=8 [((454, 252), (436, 282)), ((474, 212), (378, 356))]
P=269620, {2: 2, 5: 1, 13: 1, 17: 1, 61: 1}, Q=56412769536, {2: 8, 3: 2, 17: 1, 1049: 1, 1373: 1}, {2: 6, 17: 1, 474660713: 1} len(pairs)=8 [((718, 154), (706, 202)), ((734, 22), (686, 262))]
P=539240, {2: 3, 5: 1, 13: 1, 17: 1, 61: 1}, Q=65128699456, {2: 6, 17: 1, 313: 1, 191249: 1}, {2: 9, 11: 1, 17: 1, 14869171: 1} len(pairs)=8 [((908, 504), (872, 564)), ((948, 424), (756, 712))]
P=600445, {5: 1, 29: 1, 41: 1, 101: 1}, Q=172337340816, {2: 4, 3: 2, 37: 1, 41: 1, 337: 1, 2341: 1}, {2: 2, 41: 1, 12996866801: 1} len(pairs)=8 [((1049, 317), (953, 541)), ((1087, 139), (827, 719))]
P=606645, {3: 2, 5: 1, 13: 1, 17: 1, 61

 11%|█▏        | 277614/2460502 [00:00<00:03, 650220.30it/s]

...

 69%|██████▉   | 1707847/2460502 [00:03<00:01, 534960.96it/s]

P=10919610, {2: 1, 3: 4, 5: 1, 13: 1, 17: 1, 61: 1}, Q=26706837320676, {2: 2, 3: 8, 17: 1, 313: 1, 191249: 1}, {2: 5, 3: 8, 11: 1, 17: 1, 14869171: 1} len(pairs)=8 [((4086, 2268), (3924, 2538)), ((4266, 1908), (3402, 3204))]
P=11391445, {5: 1, 13: 3, 17: 1, 61: 1}, Q=100700319419856, {2: 4, 3: 2, 13: 4, 17: 1, 1049: 1, 1373: 1}, {2: 2, 13: 4, 17: 1, 474660713: 1} len(pairs)=16 [((4667, 1001), (4589, 1313)), ((4771, 143), (4459, 1703))]
P=10808010, {2: 1, 3: 2, 5: 1, 29: 1, 41: 1, 101: 1}, Q=60975781735716, {2: 2, 3: 4, 41: 1, 24001: 1, 191249: 1}, {2: 5, 3: 4, 41: 1, 167: 1, 733: 1, 54667: 1} len(pairs)=8 [((4482, 1236), (4098, 2196)), ((4638, 324), (3678, 2844))]


 76%|███████▌  | 1870816/2460502 [00:03<00:01, 527951.09it/s]

P=11830130, {2: 1, 5: 1, 13: 1, 17: 1, 53: 1, 101: 1}, Q=83666870549956, {2: 2, 3099713: 1, 6747953: 1}, {2: 5, 11437: 1, 2444028661: 1} len(pairs)=16 [((4648, 1434), (4506, 1832)), ((4856, 282), (4152, 2534))]


100%|██████████| 2460502/2460502 [00:04<00:00, 601432.19it/s]


In [ ]:
class GaussianIntegersParameterization:
    def __init__(self, order):
        self.order = order
        self.guide = list(itertools.product([0, 1], repeat=self.order))
        self.re_guide = [g for g in self.guide if sum(g) % 2 == 0]
        self.im_guide = [g for g in self.guide if sum(g) % 2 == 1]
        self.im_sign_matrix = (
            -(
                (
                    (
                        (np.array(self.im_guide)[:, 1:])
                        @ np.array(
                            list(itertools.product([0, 1], repeat=order - 1))
                        ).transpose()
                    )
                    + ((np.array(self.im_guide).sum(axis=1, keepdims=True) - 1) // 2)
                )
                % 2
            )
            * 2
            + 1
        )
        self.re_sign_matrix = (
            -(
                (
                    (
                        (np.array(self.re_guide)[:, 1:])
                        @ np.array(
                            list(itertools.product([0, 1], repeat=order - 1))
                        ).transpose()
                    )
                    + ((np.array(self.re_guide).sum(axis=1, keepdims=True)) // 2)
                )
                % 2
            )
            * 2
            + 1
        )

    def get_pair_group_at(self, parameters, canonize_to_first_quadrant_and_sort=False):
        assert len(parameters) == self.order
        re_monomials = [
            math.prod([p[g[i]] for i, p in enumerate(parameters)])
            for g in self.re_guide
        ]
        im_monomials = [
            math.prod([p[g[i]] for i, p in enumerate(parameters)])
            for g in self.im_guide
        ]

        im_coords = np.array(im_monomials) @ self.im_sign_matrix
        re_coords = np.array(re_monomials) @ self.re_sign_matrix

        rv = np.stack([re_coords, im_coords])
        if canonize_to_first_quadrant_and_sort:
            return np.sort(np.abs(rv), axis=0)
        else:
            return rv


gip = GaussianIntegersParameterization(4)

gip.get_pair_group_at(
    [(1, 3), (1, 2), (1, 1), (4, 5)], canonize_to_first_quadrant_and_sort=True
)

array([[40, 40, 40, 40, 16,  2, 16,  2],
       [50, 50, 50, 50, 62, 64, 62, 64]])

In [ ]:
print(sp.simplify((1 + 3 * sp.I) * (1 + 2 * sp.I) * (1 + 1 * sp.I) * (4 + 5 * sp.I)))
print(sp.simplify((1 + 3 * sp.I) * (1 + 2 * sp.I) * (1 + 1 * sp.I) * (4 - 5 * sp.I)))
print(sp.simplify((1 + 3 * sp.I) * (1 + 2 * sp.I) * (1 - 1 * sp.I) * (4 + 5 * sp.I)))
print(sp.simplify((1 + 3 * sp.I) * (1 + 2 * sp.I) * (1 - 1 * sp.I) * (4 - 5 * sp.I)))
print(sp.simplify((1 + 3 * sp.I) * (1 - 2 * sp.I) * (1 + 1 * sp.I) * (4 + 5 * sp.I)))
print(sp.simplify((1 + 3 * sp.I) * (1 - 2 * sp.I) * (1 + 1 * sp.I) * (4 - 5 * sp.I)))
print(sp.simplify((1 + 3 * sp.I) * (1 - 2 * sp.I) * (1 - 1 * sp.I) * (4 + 5 * sp.I)))
print(sp.simplify((1 + 3 * sp.I) * (1 - 2 * sp.I) * (1 - 1 * sp.I) * (4 - 5 * sp.I)))

-40 - 50*I
-40 + 50*I
-50 + 40*I
50 + 40*I
-16 + 62*I
64 + 2*I
62 + 16*I
2 - 64*I


In [13]:
np.array(im_guide)

array([[0, 0, 1],
       [0, 1, 0],
       [1, 0, 0],
       [1, 1, 1]])

In [14]:
for P, pairs in tqdm.tqdm(hits.items()):
    handle_pairs(pairs)

  2%|▏         | 48925/2460502 [00:00<00:04, 489105.87it/s]

8 67405 {5: 1, 13: 1, 17: 1, 61: 1} 3525798096 (((343, 131), (367, 11)), ((353, 101), (359, 77)))
8 134810 {2: 1, 5: 1, 13: 1, 17: 1, 61: 1} 4070543716 (((378, 356), (474, 212)), ((436, 282), (454, 252)))
8 269620 {2: 2, 5: 1, 13: 1, 17: 1, 61: 1} 56412769536 (((686, 262), (734, 22)), ((706, 202), (718, 154)))


  6%|▌         | 136228/2460502 [00:00<00:07, 311434.87it/s]

8 539240 {2: 3, 5: 1, 13: 1, 17: 1, 61: 1} 65128699456 (((756, 712), (948, 424)), ((872, 564), (908, 504)))
8 600445 {5: 1, 29: 1, 41: 1, 101: 1} 172337340816 (((827, 719), (1087, 139)), ((953, 541), (1049, 317)))
8 606645 {3: 2, 5: 1, 13: 1, 17: 1, 61: 1} 285589645776 (((1029, 393), (1101, 33)), ((1059, 303), (1077, 231)))


  9%|▉         | 224462/2460502 [00:00<00:09, 239033.96it/s]

...

 71%|███████   | 1749816/2460502 [00:13<00:07, 95259.42it/s]

8 10808010 {2: 1, 3: 2, 5: 1, 29: 1, 41: 1, 101: 1} 60975781735716 (((3678, 2844), (4638, 324)), ((4098, 2196), (4482, 1236)))


 74%|███████▎  | 1809763/2460502 [00:14<00:06, 94771.75it/s]

16 11830130 {2: 1, 5: 1, 13: 1, 17: 1, 53: 1, 101: 1} 83666870549956 (((4152, 2534), (4856, 282)), ((4506, 1832), (4648, 1434)))


100%|██████████| 2460502/2460502 [00:16<00:00, 152438.65it/s]


# Gaussian integers

In [137]:
def find_gaussian_prime_rep(p):
    assert p % 4 == 1 or p == 2
    for x in range(int(math.sqrt(p)), 0, -1):
        y, valid = sp.integer_nthroot((p - x**2), 2)
        if valid:
            assert x >= y, (x, y)
            return sp.ZZ_I(x, y)
    return None


print(find_gaussian_prime_rep(2))


B = 10000

gaussian_primes = [2] + [p for p in sp.primerange(B) if p % 4 == 1]
gaussian_primes_reps = {p: find_gaussian_prime_rep(p) for p in gaussian_primes}
# gaussian_primes_reps

1 + I


In [15]:
def get_pairs_from_factors(factors):
    EFFICIENCY_SKIP = 2
    assert 2 in factors, "P must be divided by 2"
    reps = [gaussian_primes_reps[p] for p in factors]
    pairs = set()
    for mask in itertools.product([0, 1], repeat=len(factors) - EFFICIENCY_SKIP):
        prod = sp.prod(reps[:EFFICIENCY_SKIP])
        for i, bit in enumerate(mask):
            prod *= (
                reps[EFFICIENCY_SKIP + i]
                if bit
                else sp.ZZ_I(reps[EFFICIENCY_SKIP + i].x, -reps[EFFICIENCY_SKIP + i].y)
            )
        pairs.add(tuple(sorted([abs(prod.x), abs(prod.y)])))
    return list(pairs)


pairs = get_pairs_from_factors(gaussian_primes[:4])
handle_pairs_2(pairs)

In [16]:
!factor 1685125

1685125: 5 5 5 13 17 61


In [17]:
handle_pairs_2(get_pairs_from_factors([2, 5, 5, 5, 13, 17, 61]))

P=1685125, Q=2203623810000, len(pairs)=16, {5: 3, 13: 1, 17: 1, 61: 1} [((385, 1795), (505, 1765)), ((655, 1715), (55, 1835))]


In [18]:
get_pairs_from_factors([2, 5, 5, 5, 13, 17, 61])

[(1109, 1463),
 (1145, 1435),
 (505, 1765),
 (133, 1831),
 (335, 1805),
 (1205, 1385),
 (827, 1639),
 (1057, 1501),
 (979, 1553),
 (769, 1667),
 (461, 1777),
 (1243, 1351),
 (55, 1835),
 (815, 1645),
 (385, 1795),
 (655, 1715)]

In [19]:
len(get_pairs_from_factors([2, 17, 17, 17, 17, 17, 29]))

6

In [20]:
handle_pairs_2(get_pairs_from_factors(gaussian_primes[:13]))

In [31]:
sp.factorint(4179412643232853), sp.factorint(9065468187959341516904114403984)

({13: 1, 17: 1, 29: 2, 61: 1, 109: 1, 113: 1, 173: 2},
 {2: 4, 3: 2, 17: 1, 29: 4, 173: 4, 313: 1, 677: 1, 27584773: 1})

In [43]:
import random


jobs = list(itertools.combinations_with_replacement(gaussian_primes[0:17], 7))
# jobs = random.shuffle(jobs)
# jobs = random.choice(jobs, 10000)

# jobs = [random.choices(gaussian_primes[0:25], k=6) for _ in range(100000)]

# jobs
for factors in tqdm.tqdm(jobs):
    handle_pairs_2(get_pairs_from_factors([2, *factors]))
    # get_pairs_from_factors([2, *factors])

  1%|          | 1354/245157 [00:00<01:51, 2183.80it/s]

P=539240, {2: 3, 5: 1, 13: 1, 17: 1, 61: 1}, Q=65128699456, {2: 6, 17: 1, 313: 1, 191249: 1}, {2: 9, 11: 1, 17: 1, 14869171: 1} len(pairs)=8 [((712, 756), (424, 948)), ((564, 872), (504, 908))]
P=4803560, {2: 3, 5: 1, 29: 1, 41: 1, 101: 1}, Q=12044598861376, {2: 6, 41: 1, 24001: 1, 191249: 1}, {2: 9, 41: 1, 167: 1, 733: 1, 54667: 1} len(pairs)=8 [((216, 3092), (1896, 2452)), ((824, 2988), (1464, 2732))]


  3%|▎         | 6186/245157 [00:02<01:48, 2198.10it/s]

P=23660260, {2: 2, 5: 1, 13: 1, 17: 1, 53: 1, 101: 1}, Q=225140421067776, {2: 10, 3: 2, 541: 1, 821: 1, 55001: 1}, {2: 6, 1693: 1, 2917: 1, 9934081: 1} len(pairs)=16 [((2674, 6338), (3214, 6082)), ((4574, 5138), (1618, 6686))]


  4%|▍         | 10136/245157 [00:04<01:48, 2167.73it/s]

...

 60%|██████    | 148130/245157 [01:25<01:06, 1462.20it/s]

P=617855219917, {13: 1, 17: 1, 61: 3, 109: 1, 113: 1}, Q=198122214741906723621264, {2: 4, 3: 2, 17: 1, 61: 4, 313: 1, 677: 1, 27584773: 1}, {2: 2, 11: 1, 17: 1, 61: 4, 223958558073259: 1} len(pairs)=32 [((294325, 1071953), (529297, 977525)), ((606035, 931897), (200995, 1093303))]
P=884856346933, {13: 1, 17: 1, 61: 1, 73: 2, 109: 1, 113: 1}, Q=406354688147467527390864, {2: 4, 3: 2, 17: 1, 73: 4, 313: 1, 677: 1, 27584773: 1}, {2: 2, 11: 1, 17: 1, 73: 4, 223958558073259: 1} len(pairs)=48 [((240535, 1308379), (725255, 1115221)), ((633421, 1169825), (352225, 1282829))]
P=1315246223317, {13: 1, 17: 1, 61: 1, 89: 2, 109: 1, 113: 1}, Q=897788133258966678366864, {2: 4, 3: 2, 17: 1, 89: 4, 313: 1, 677: 1, 27584773: 1}, {2: 2, 11: 1, 17: 1, 89: 4, 223958558073259: 1} len(pairs)=48 [((293255, 1595147), (884215, 1359653)), ((429425, 1563997), (772253, 1426225))]


 61%|██████    | 148440/245157 [01:25<01:06, 1452.91it/s]

P=1562321893093, {13: 1, 17: 1, 61: 1, 97: 2, 109: 1, 113: 1}, Q=1266778754806486858491024, {2: 4, 3: 2, 17: 1, 97: 4, 313: 1, 677: 1, 27584773: 1}, {2: 2, 11: 1, 17: 1, 97: 4, 223958558073259: 1} len(pairs)=48 [((963695, 1481869), (319615, 1738531)), ((468025, 1704581), (841669, 1554425))]
P=1693829910877, {13: 1, 17: 1, 61: 1, 101: 2, 109: 1, 113: 1}, Q=1489015879429131474543504, {2: 4, 3: 2, 17: 1, 101: 4, 313: 1, 677: 1, 27584773: 1}, {2: 2, 11: 1, 17: 1, 101: 4, 223958558073259: 1} len(pairs)=48 [((487325, 1774873), (876377, 1618525)), ((332795, 1810223), (1003435, 1542977))]
P=1972786312237, {13: 1, 17: 1, 61: 1, 109: 3, 113: 1}, Q=2019853289245098419078544, {2: 4, 3: 2, 17: 1, 109: 4, 313: 1, 677: 1, 27584773: 1}, {2: 2, 11: 1, 17: 1, 109: 4, 223958558073259: 1} len(pairs)=32 [((359155, 1953607), (1082915, 1665193)), ((525925, 1915457), (945793, 1746725))]
P=2120234695813, {13: 1, 17: 1, 61: 1, 109: 1, 113: 3}, Q=2333069133839048664115344, {2: 4, 3: 2, 17: 1, 113: 4, 313: 1, 677

100%|██████████| 245157/245157 [02:25<00:00, 1686.21it/s]


In [26]:
# from concurrent.futures import ThreadPoolExecutor

# jobs = [random.choices(gaussian_primes[0:20], k=8) for _ in range(100000)]


# def your_function(x):
#     handle_pairs_2(get_pairs_from_factors([2, *x]))


# with ThreadPoolExecutor(4) as executor:
#     executor.map(your_function, tqdm.tqdm(jobs))

In [ ]:
for factors in tqdm.tqdm(
    list(itertools.combinations_with_replacement(gaussian_primes[0:13], 7))
):
    handle_pairs_2(get_pairs_from_factors([2, *factors]))

[5, 13, 17, 29, 37, 41, 53, 61]

In [30]:
from tqdm.contrib.concurrent import thread_map

job = lambda x: handle_pairs(get_pairs_from_factors([2, *x]))
thread_map(
    job,
    # list(itertools.combinations_with_replacement(gaussian_primes[0:13], 7)),
    jobs,
    max_workers=5,
)

  0%|          | 0/100000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [141]:
gaussian_primes[400]

6277

In [152]:
factorsa = []
factorsb = []
count = 0

# factors = [2, 5, 13,13, 17, 29, 37, 61, 73, 89, 181, 2797, 6277]
factors = [2, 13, 17]
P = sp.prod(factors)
print(P % 3)
print(P % 7)
print(P % 11)
print(P)
for a, b in get_pairs_from_factors(factors):
    factorsa.extend(sp.factorint(a).keys())
    factorsb.extend(sp.factorint(b).keys())
    count += 1
    print(sp.factorint(a), sp.factorint(b))
print(f"{count=}")
print(
    sorted(
        list(collections.Counter(factorsa).items())[:10],
        key=lambda x: x[1],
        reverse=True,
    )
)
print(
    sorted(
        list(collections.Counter(factorsb).items())[:10],
        key=lambda x: x[1],
        reverse=True,
    )
)

1
1
2
442
{3: 2} {19: 1}
{} {3: 1, 7: 1}
count=2
[(3, 1)]
[(19, 1), (3, 1), (7, 1)]


In [ ]:
{11: 1} {3: 1, 5: 2}
{3: 2, 5: 1} {61: 1}
{3: 1, 13: 1} {5: 1, 13: 1}


{13: 1} {3: 1, 7: 1, 13: 1}
{3: 1, 61: 1} {7: 1, 29: 1}
{3: 1, 31: 1} {257: 1}
{3: 2, 13: 1} {13: 1, 19: 1}

ZZ_I(0, 4)

In [ ]:
5 * 13 * 17 * 29 * 37  # * 41 * 53

1185665

In [ ]:
650 = 13 * 5 * 5 * 2

In [ ]:
# Don't forget that the real sum (norm) is 2 times the key (which is P).
print(len(hits[5 * 13]), hits[5 * 13])
print(len(hits[5 * 13 * 17]), hits[5 * 13 * 17])
print(len(hits[5 * 13 * 17 * 29]), hits[5 * 13 * 17 * 29])
print(len(hits[5 * 13 * 17 * 29 * 37]), hits[5 * 13 * 17 * 29 * 37])
print(len(hits[5 * 13 * 17 * 29 * 37 * 41]), hits[5 * 13 * 17 * 29 * 37 * 41])

32 [(7031, 6913), (7103, 6839), (7483, 6421), (7529, 6367), (7571, 6317), (7793, 6041), (7907, 5891), (8137, 5569), (8377, 5201), (8431, 5113), (8479, 5033), (8743, 4559), (8773, 4501), (8893, 4259), (8929, 4183), (9149, 3677), (9161, 3647), (9187, 3581), (9331, 3187), (9377, 3049), (9457, 2791), (9517, 2579), (9611, 2203), (9653, 2011), (9719, 1663), (9733, 1579), (9749, 1477), (9803, 1061), (9839, 647), (9847, 511), (9851, 427), (9859, 157)]
16 [(1151, 1023), (1187, 981), (1243, 909), (1263, 881), (1299, 827), (1333, 771), (1341, 757), (1387, 669), (1441, 543), (1473, 449), (1497, 361), (1509, 307), (1511, 297), (1527, 199), (1529, 183), (1539, 53)]
8 [(181, 177), (197, 159), (219, 127), (233, 99), (237, 89), (243, 71), (251, 33), (253, 9)]
4 [(37, 29), (41, 23), (43, 19), (47, 1)]
2 [(9, 7), (11, 3)]


In [292]:
list(range(10, 1, -1))

[10, 9, 8, 7, 6, 5, 4, 3, 2]

In [259]:
hits[18785]

[(141, 133), (153, 119), (177, 79), (187, 51), (189, 43), (191, 33)]

In [287]:
[p for p in sp.primerange(200) if p % 4 == 1]

[5,
 13,
 17,
 29,
 37,
 41,
 53,
 61,
 73,
 89,
 97,
 101,
 109,
 113,
 137,
 149,
 157,
 173,
 181,
 193,
 197]